# Preprocess, Segment, Train, and Validate

Select raw training data from any run folder, preprocess it, build sliding windows, train the LSTM, and save training artifacts to the output run folder.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'training.py').exists():
    ROOT = Path(r'D:/BME/BCI/online_bci/online_eeg')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from preprocessing import (
    AudioLabelConfig,
    PreprocessConfig,
    labeled_preprocess_summary,
    preprocess_recording,
)
from training import TrainingConfig, WindowConfig, train_validate_pipeline
from plots import plot_labeled_recording

print('Pipeline root:', ROOT)

## Training Settings

In [ ]:
RUNS_ROOT = ROOT / 'runs'

TRAIN_RUN_IDS = ('run_001',)
EXTRA_TRAIN_RAW_NPZ = ()
RAW_SUBDIR = 'raw_training'
RAW_PATTERN = '*.npz'

OUTPUT_RUN_ID = 'run_001'
OUTPUT_RUN_DIR = RUNS_ROOT / OUTPUT_RUN_ID
LABELED_DIR = OUTPUT_RUN_DIR / 'labeled_training'

EEG_CHANNELS = (1, 2, 3, 4)
EOG_CHANNELS = (5,)
EEG_CHANNEL_NAMES = ('O1', 'Oz', 'O2', 'POz')
AUDIO_CHANNEL = 16

APPLY_SOFTWARE_FILTERS = False  # BIOPAC hardware already bandpasses the EEG at 1-35 Hz.
SOFTWARE_BANDPASS_HZ = None  # Set to (1.0, 35.0) only if APPLY_SOFTWARE_FILTERS=True.
SOFTWARE_NOTCH_HZ = None  # Set to (60.0,) only if APPLY_SOFTWARE_FILTERS=True.
DEMEAN_CHANNELS = True

SLIDING_WINDOW_SEC = 2.0
SLIDING_STRIDE_SEC = 0.05
LABEL_MODE = 'endpoint'  # use 'majority' to label each window by the most common sample label.

PRE = PreprocessConfig(
    eeg_channels=EEG_CHANNELS,
    eog_channels=EOG_CHANNELS,
    audio_channel=AUDIO_CHANNEL,
    apply_software_filters=APPLY_SOFTWARE_FILTERS,
    bandpass_low_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[0],
    bandpass_high_hz=None if SOFTWARE_BANDPASS_HZ is None else SOFTWARE_BANDPASS_HZ[1],
    notch_hz=SOFTWARE_NOTCH_HZ,
    notch_quality_factor=30.0,
    filter_order=4,
    demean_channels=DEMEAN_CHANNELS,
)

LABELS = AudioLabelConfig(
    class_names=('Eyes Open', 'Eyes Closed'),
    baseline_label=0,
    active_label=1,
    cue_label_sequence=None,
    alternate_binary_labels=True,
    label_duration_sec=None,  # transition mode: each cue switches state until the next cue.
    label_start_offset_sec=0.0,  # label switch starts exactly at cue onset.
    envelope_window_sec=0.025,
    onset_threshold=None,
    onset_min_interval_sec=0.50,
)

WIN = WindowConfig(
    feature_mode='filtered_signal',
    window_sec=SLIDING_WINDOW_SEC,
    stride_sec=SLIDING_STRIDE_SEC,
    label_mode=LABEL_MODE,
)

TRAIN = TrainingConfig(
    train_fraction=1.0,
    hidden_size=64,
    num_layers=2,
    dropout=0.2,
    batch_size=64,
    epochs=20,
    lr=1e-3,
    seed=888,
)

OUTPUT_RUN_DIR.mkdir(parents=True, exist_ok=True)
LABELED_DIR.mkdir(parents=True, exist_ok=True)
print('Training data run IDs:', TRAIN_RUN_IDS)
print('Output run directory:', OUTPUT_RUN_DIR)
print('Labeled output directory:', LABELED_DIR)

## Resolve Raw Training Data

In [ ]:
def _safe_stem(value):
    text = str(value).strip().replace(' ', '_')
    for char in ':/\\':
        text = text.replace(char, '_')
    return text or 'source'

raw_records = []
missing_raw_dirs = []
for run_id in TRAIN_RUN_IDS:
    raw_dir = RUNS_ROOT / str(run_id) / RAW_SUBDIR
    run_paths = sorted(path for path in raw_dir.glob(RAW_PATTERN) if path.is_file())
    if not run_paths:
        missing_raw_dirs.append(raw_dir)
    raw_records.extend((str(run_id), path) for path in run_paths)

for raw_path in EXTRA_TRAIN_RAW_NPZ:
    raw_records.append(('extra', Path(raw_path)))

if missing_raw_dirs:
    raise FileNotFoundError('No raw training NPZ files found in: ' + ', '.join(str(path) for path in missing_raw_dirs))
if not raw_records:
    raise ValueError('No raw training data selected. Set TRAIN_RUN_IDS or EXTRA_TRAIN_RAW_NPZ.')

raw_table = pd.DataFrame(
    [{'source_run_id': run_id, 'raw_npz': str(path)} for run_id, path in raw_records]
)
display(raw_table)
raw_records

## Preprocess and Audio-Channel Labels

In [ ]:
labeled_paths = []
cue_tables = {}
for source_run_id, raw_path in raw_records:
    raw_path = Path(raw_path)
    labeled_name = f'{_safe_stem(source_run_id)}__{raw_path.stem}_labeled.npz'
    labeled_path = LABELED_DIR / labeled_name
    labeled_npz, cue_table = preprocess_recording(
        raw_npz=raw_path,
        output_npz=labeled_path,
        preprocess_config=PRE,
        label_config=LABELS,
    )
    labeled_paths.append(labeled_npz)
    cue_tables[str(labeled_npz)] = cue_table

display(labeled_preprocess_summary(labeled_paths))
labeled_paths

In [ ]:
fig, axes = plot_labeled_recording(
    labeled_paths[0],
    max_duration_sec=None,
    channel_names=EEG_CHANNEL_NAMES,
)
axes[0].set_title('First labeled training recording preview')

## Train and Validate LSTM

In [ ]:
train_result = train_validate_pipeline(
    labeled_npz_paths=labeled_paths,
    output_dir=OUTPUT_RUN_DIR,
    window_config=WIN,
    training_config=TRAIN,
)

checkpoint_path = train_result['checkpoint_path']
print('Saved model checkpoint:', checkpoint_path)
print('Aligned validation EEG/prediction CSV:', train_result['validation_aligned_prediction_csv'])
checkpoint_path

In [ ]:
history = train_result['history']
display(history.tail())
display(train_result['validation_summary'])
display(train_result['validation_per_class'])

ax = history.plot(x='epoch', y=['train_acc', 'val_acc'], marker='o', figsize=(8, 4))
ax.set_ylabel('Accuracy')
ax.set_title('Training and validation accuracy')
ax.grid(True, alpha=0.3)
plt.tight_layout()